# NHANES project about oral frailty: descriptive and regression analysis
> This notebook has the purpose to collect all the analysis on Nhanes dataset for a medical paper project 

Requirements and Information:
1. Nhanes dataset from 1999/00 to 2001/02
2. Oral Frailty Index with:
    - 1: Do you have any difficulties eating tough foods compared to 6 months ago? (OHQ080)
    - 2: Have you choked on your tea or soup recently? (OHQ100 and OHQ105)
    - 3: Do you use dentures? (OHXEDEN)
    - 4: Do you often have a dry mouth?	(OHQ110)
    - 5: Do you go out less frequently than you did last year? (PAQ500)
    - 6: Can you eat hard foods like squid jerky or pickled radish?	(OHQ020)
    - 7: How many times do you brush your teeth in a day? (3 or more times/day)	(OHQ040 and OHQ010)
    - 8:  Do you visit a dental clinic at least annually? (OHQ050)
3. Outcome:
    - TO BE DEFINED
4. Demographic Data:
    - Gender (RIAGENDR)
    - Age at screening (RIDAGEYR)
    - Race (RIDRETH1)
    - Education	(DMDEDUC2)
    - Poverty income ratio (INDFMPIR)
    - Smoking status (SMQ020)
5. Confounding Variables:
    - Heart failure	(RIDRETH1)  
    - Coronary heart disease (MCQ160b)
    - Stroke (MCQ160c)
    - COPD (MCQ160f)
    - Liver disease	(MCQ160o)
    - Cancer (MCQ500)
    - Diabetes (MCQ220)
    - High blood pressure (DIQ010)
6. Age => 60 
7. Oral Frailty Index cutoff: 
8. Oral Frailty Index groups:

## Import libraries

Reference for nhanesA [here](https://cran.r-project.org/web/packages/nhanesA/nhanesA.pdf)

In [ ]:
library(haven)
library(nhanesA)
library(survey)
library(MASS)
library(dplyr)
library(tidyr)
library(tidyverse)
library(ggplot2)
library(readr)
library(flextable)
library(officer)
library(nnet)
library(broom)
library(ggplot2)

## Configurations

In [ ]:
path_to_data_99_00 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/1999_00/"
path_to_data_01_02 <- "/Users/silvanoquarto/Desktop/LAVORO/MEDICAL_PHYSICS/Med-Physics/data/NHANES/2001_02/"

## Load NHANES data (1999/00-2001/02)

In [ ]:
# Datasets for 1999/00 period

demo_99_00 <- read_xpt(file.path(path_to_data_99_00, "DEMO.xpt"))

demo_99_00_selected <- demo_99_00 %>%
  select(SEQN, RIAGENDR, RIDAGEYR, RIDRETH1, DMDEDUC2, INDFMPIR)

alcohol_99_00 <- read_xpt(file.path(path_to_data_99_00, "ALQ.xpt"))

alcohol_99_00_selected <- alcohol_99_00 %>%
  select(SEQN, ALQ101)

smoking_99_00 <- read_xpt(file.path(path_to_data_99_00, "SMQ.xpt"))

smoking_99_00_selected <- smoking_99_00 %>%
    select(SEQN, SMQ020)

med_conditions_99_00 <- read_xpt(file.path(path_to_data_99_00, "MCQ.xpt"))

med_conditions_99_00_selected <- med_conditions_99_00 %>%
    select(SEQN, MCQ140, MCQ160B, MCQ160C, MCQ160D, MCQ160E, MCQ160F, MCQ160L, MCQ220)

med_conditions_99_00_selected <- med_conditions_99_00_selected %>%
  rename(DLQ020 = MCQ140)


blood_pressure_99_00 <- read_xpt(file.path(path_to_data_99_00, "BPQ.xpt"))

blood_pressure_99_00_selected <- blood_pressure_99_00 %>%
    select(SEQN, BPQ020)


diabetes_99_00 <- read_xpt(file.path(path_to_data_99_00, "DIQ.xpt"))

diabetes_99_00_selected <- diabetes_99_00 %>%
    select(SEQN, DIQ010)


oral_99_00_selected <- oral_99_00 %>%
    select(SEQN, OHQ080, OHQ100, OHQ110, OHQ020, OHQ040, OHQ050)

In [ ]:
# Datasets for 2001/02 period

oral_01_02_selected <- oral_01_02 %>%
    select(SEQN, OHQ085, OHQ105, OHQ115, OHQ020, OHQ040, OHQ050)

## Data Information

In [ ]:
# Function to show dataset information

print_dataset_info <- function(df, name) {
    cat(sprintf("\nDataset: %s\n", name))
    cat(sprintf("Number of rows: %d\n", nrow(df)))
    cat(sprintf("Number of columns: %d\n", ncol(df)))
    cat(sprintf("Memory usage: %.2f MB\n", object.size(df) / 1024^2))
    cat("\nColumn details:\n")
    for (col_name in colnames(df)) {
        cat(sprintf("- %s: %s\n", col_name, class(df[[col_name]])))
    }
}


In [ ]:
print_dataset_info(demo_99_00_selected, "Demographics 1999-2000")

In [ ]:
print_dataset_info(demo_01_02_selected, "Demographics 2001-2002")

In [ ]:
print_dataset_info(smoking_99_00_selected, "Smoking Status 1999-2000")

In [ ]:
print_dataset_info(smoking_01_02_selected, "Smoking Status 2001-2002")

In [ ]:
print_dataset_info(med_cond_99_00_selected, "Medical Conditions 1999-2000")

In [ ]:
print_dataset_info(med_cond_01_02_selected, "Medical Conditions 2001-2002")

In [ ]:
print_dataset_info(diabetes_99_00_selected, "Diabetes 1999-2000")

In [ ]:
print_dataset_info(diabetes_01_02_selected, "Diabetes 2001-2002")

In [ ]:
print_dataset_info(blood_pressure_99_00_selected, "High Blood Pressure 1999-2000")

In [ ]:
print_dataset_info(blood_pressure_01_02_selected, "High Blood Pressure 2001-2002")

In [ ]:
print_dataset_info(physical_99_00_selected, "Physical Activities 1999-2000")

In [ ]:
print_dataset_info(physical_01_02_selected, "Physical Activities 2001-2002")

In [ ]:
print_dataset_info(eden_99_00_selected, "Edentulous 1999-2000")

In [ ]:
print_dataset_info(eden_01_02_selected, "Edentulous 2001-2002")

In [ ]:
print_dataset_info(oral_99_00_selected, "Oral Health 1999-2000")

In [ ]:
print_dataset_info(oral_01_02_selected, "Oral Health 2001-2002")

### Unique values and % for each categorical feature in the datasets

In [ ]:
# Function to see unique values for categorical features

print_categorical_info <- function(df, name) {
  cat(sprintf("\nCategorical Column Analysis for Dataset: %s\n", name))

  categorical_cols <- names(df)[sapply(df, is.factor) | sapply(df, is.character)]

  if (length(categorical_cols) == 0) {
    cat("No categorical columns found.\n")
    return()
  }

  for (col_name in categorical_cols) {
    cat(sprintf("\nColumn: %s\n", col_name))
    value_counts <- table(df[[col_name]])  
    total_count <- sum(value_counts)      
    percentages <- round(100 * value_counts / total_count, 2) 

    for (i in seq_along(value_counts)) {
      cat(sprintf("- %s: %d (%.2f%%)\n", names(value_counts)[i], value_counts[i], percentages[i]))
    }
  }
}

In [ ]:
print_categorical_info(demo_99_00_selected, "Demographics 1999-2000")

In [ ]:
print_categorical_info(demo_01_02_selected, "Demographics 2001-2002")

In [ ]:
print_categorical_info(smoking_99_00_selected, "Smoking Status 1999-2000")

In [ ]:
print_categorical_info(smoking_01_02_selected, "Smoking Status 2001-2002")

In [ ]:
print_categorical_info(med_cond_99_00_selected, "Medical Conditions 1999-2000")

In [ ]:
print_categorical_info(med_cond_01_02_selected, "Medical Conditions 2001-2002")

In [ ]:
print_categorical_info(diabetes_99_00_selected, "Diabetes 1999-2000")

In [ ]:
print_categorical_info(diabetes_01_02_selected, "Diabetes 2001-2002")

In [ ]:
print_categorical_info(blood_pressure_99_00_selected, "High Blood Pressure 1999-2000")

In [ ]:
print_categorical_info(blood_pressure_01_02_selected, "High Blood Pressure 2001-2002")

In [ ]:
print_categorical_info(physical_99_00_selected, "Physical Activities 1999-2000")

In [ ]:
print_categorical_info(physical_01_02_selected, "Physical Activities 2001-2002")

In [ ]:
print_categorical_info(eden_99_00_selected, "Edentulous 1999-2000")

In [ ]:
print_categorical_info(eden_01_02_selected, "Edentulous 2001-2002")

In [ ]:
print_categorical_info(oral_99_00_selected, "Oral Health 1999-2000")

In [ ]:
print_categorical_info(oral_01_02_selected, "Oral Health 2001-2002")

### Missing Values and % for each dataset

In [ ]:
# Function to calculate missing values

missing_values <- function(df) {
  missing_count <- colSums(is.na(df))
  missing_percent <- (missing_count/nrow(df)) * 100

  missing_df <- data.frame(
    variable = names(missing_count),
    n_missing = missing_count,
    percent_missing = round(missing_percent, 2)
  ) %>%
    arrange(desc(n_missing))

  return(missing_df)
}

In [ ]:
demo_99_00_missing <- missing_values(demo_99_00_selected)
print(demo_99_00_missing)

In [ ]:
demo_01_02_missing <- missing_values(demo_01_02_selected)
print(demo_01_02_missing)

In [ ]:
smoking_99_00_missing <- missing_values(smoking_99_00_selected)
print(smoking_99_00_missing)

In [ ]:
smoking_01_02_missing <- missing_values(smoking_01_02_selected)
print(smoking_01_02_missing)

In [ ]:
med_cond_99_00_missing <- missing_values(med_cond_99_00_selected)
print(med_cond_99_00_missing)

In [ ]:
med_cond_01_02_missing <- missing_values(med_cond_01_02_selected)
print(med_cond_01_02_missing)

In [ ]:
diabetes_99_00_missing <- missing_values(diabetes_99_00_selected)
print(diabetes_99_00_missing)

In [ ]:
diabetes_01_02_missing <- missing_values(diabetes_01_02_selected)
print(diabetes_01_02_missing)

In [ ]:
blood_pressure_99_00_missing <- missing_values(blood_pressure_99_00_selected)
print(blood_pressure_99_00_missing)

In [ ]:
blood_pressure_01_02_missing <- missing_values(blood_pressure_01_02_selected)
print(blood_pressure_01_02_missing)

In [ ]:
physical_99_00_missing <- missing_values(physical_99_00_selected)
print(physical_99_00_missing)

In [ ]:
physical_01_02_missing <- missing_values(physical_01_02_selected)
print(physical_01_02_missing)

In [ ]:
eden_99_00_missing <- missing_values(eden_99_00_selected)
print(eden_99_00_missing)

In [ ]:
eden_01_02_missing <- missing_values(eden_01_02_selected)
print(eden_01_02_missing)

In [ ]:
oral_99_00_missing <- missing_values(oral_99_00_selected)
print(oral_99_00_missing)

In [ ]:
oral_01_02_missing <- missing_values(oral_01_02_selected)
print(oral_01_02_missing)

#### Merge all df with missing data information for each period and plot results

In [ ]:
missing_99_00 <- bind_rows(
  oral_99_00_missing, eden_99_00_missing, physical_99_00_missing,
  blood_pressure_99_00_missing, diabetes_99_00_missing, med_cond_99_00_missing,
  smoking_99_00_missing, demo_99_00_missing
) %>%
  mutate(period = "1999-2000")

missing_01_02 <- bind_rows(
  oral_01_02_missing, eden_01_02_missing, physical_01_02_missing,
  blood_pressure_01_02_missing, diabetes_01_02_missing, med_cond_01_02_missing,
  smoking_01_02_missing, demo_01_02_missing
) %>%
  mutate(period = "2001-2002")

In [ ]:
missing_data <- bind_rows(missing_99_00, missing_01_02)

missing_data <- missing_data %>%
mutate(variable_label = paste0(variable, " (", round(percent_missing, 1), "%)"))

In [ ]:
options(repr.plot.width=15, repr.plot.height=12)

ggplot(missing_data, aes(x = reorder(variable_label, -n_missing), y = n_missing, fill = period)) + 
  geom_col(position = 'dodge') + 
  coord_flip() + 
  labs(title = "Missing Data for each Variable",
       x = "Variables",
       y = "Number of missing values") + 
  theme_minimal() + 
  theme(
    plot.title = element_text(size = 22, face = "bold"),  
    axis.text.y = element_text(size = 16, face = "bold"),  
    axis.text.x = element_text(size = 16),
    axis.title.x = element_text(size = 14),
    axis.title.y = element_text(size = 14),
    legend.text = element_text(size = 16),
    legend.title = element_text(size = 16)
  ) +
  scale_fill_manual(values = c("1999-2000" = "#E69F00", "2001-2002" = "#56B4E9"))

## Merge datasets in a one and complete data frame 

Steps:
1. Horizontal union for the period 1999-2000
2. Horizontal union for the period 2001-2002
3. Vertical union of the two periods
4. Filter with Age >= 60

In [ ]:
# Rename the following features that are the same of each df but with different names:
# OHQ080 e OHQ085
# OHQ100 e OHQ105
# OHQ0110 e OHQ115

oral_01_02_selected_renamed <- oral_01_02_selected %>%
  rename(OHQ080 = OHQ085, OHQ100 = OHQ105, OHQ110 = OHQ115)

colnames(oral_01_02_selected_renamed)

In [ ]:
datasets_99_00 <- list(
  demo_99_00_selected, smoking_99_00_selected, med_cond_99_00_selected,
  diabetes_99_00_selected, blood_pressure_99_00_selected, physical_99_00_selected,
  eden_99_00_selected, oral_99_00_selected
)

datasets_01_02 <- list(
  demo_01_02_selected, smoking_01_02_selected, med_cond_01_02_selected,
  diabetes_01_02_selected, blood_pressure_01_02_selected, physical_01_02_selected,
  eden_01_02_selected, oral_01_02_selected_renamed
)

# Horizontal union for period 1999/00 and 2001/02

df_99_00 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_99_00)

df_01_02 <- Reduce(function(x, y) full_join(x, y, by = "SEQN"), datasets_01_02)

In [ ]:
# Vertical union for these two periods

df_final <- bind_rows(df_99_00, df_01_02)

dim(df_final)

In [ ]:
# Condition to exclude people younger than 60 years old

elderly_df_final <- df_final %>%
  filter(RIDAGEYR >= 60)

dim(elderly_df_final)

In [ ]:
# Remove every row with a least one NA values

elderly_df_final_no_na <- na.omit(elderly_df_final)

dim(elderly_df_final_no_na)

In [ ]:
# Other method to choose which NA columns to drop

elderly_df_final_no_na <- elderly_df_final %>% drop_na()

dim(elderly_df_final_no_na)

In [ ]:
# JUST FOR TESTING

elderly_df_final_no_eden_ohq040 <- subset(elderly_df_final, select = -c(OHXEDEN, OHQ040))

elderly_df_final_no_eden_ohq040 <- na.omit(elderly_df_final_no_eden_ohq040)

dim(elderly_df_final_no_eden_ohq040)